# Recreating ALE VIM Experiments

- Example 4. Theoretical Linear Model with Correlation
- Example 5. Uniform DGP with Squared Term and Correlation on NN
- Example 6. Gaussian Copula DGP with Dummy and Correlation on NN, GBT, and RF
- Example 7. Real-world Bike Sharing Data on NN

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline

from algorithms.shared import bin_selection
from algorithms.ale import ale_1d
from algorithms.ale_vim import ale_main_vim, ale_connected_total, ale_quantile_total
from utils import replicate_ale_vims, replicate_ale_vim_training

## Example 4

In [ ]:
def example_4_dgp(n, rho):
    mean = [0, 0, 0]
    # notice X3 is independent of X1 and X2
    cov = [[1, rho, 0], [rho, 1, 0], [0, 0, 1]]
    X = np.random.multivariate_normal(mean, cov, n)
    return X

def example_4_f(X, betas):
    return betas[0] * X[:, 0] + betas[1] * X[:, 1] + betas[2] * X[:, 2]

n = 1000
dgp = lambda n: example_4_dgp(n, rho=0.5)
f = lambda X: example_4_f(X, betas=[3, 2, 1])
bins = bin_selection(n)

vim_mains, vim_connected, vim_quantile = replicate_ale_vims(dgp, f, 3, n=n, bins=bins, replications=5, categorical=False)


## Example 5

In [ ]:
def example_5_dgp(n):
    u = np.random.uniform(0, 1, n)
    x1 = u + np.random.normal(0, 0.05, n)
    x2 = u + np.random.normal(0, 0.05, n)
    X = np.column_stack((x1, x2))
    y = x1 + x2 ** 2 + np.random.normal(0, 0.1, n)
    return X, y

In [ ]:
# # determine optimal hyperparameters for MLPRegressor
# def get_mlp_hyperparams(X, y):
#     # use GridSearchCV to find the best hyperparameters
#     param_grid = {
#         'hidden_layer_sizes': [(x, ) for x in np.arange(50, 101, 5)],
#         'activation': ['relu'],
#         'solver': ['adam'],
#         'alpha': 0.00001 * np.arange(1, 11),
#         'learning_rate': ['constant']
#     }
#     mlp = MLPRegressor(max_iter=1000, random_state=42)
#     grid_search = GridSearchCV(mlp, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
#     grid_search.fit(X, y)
#     return grid_search.best_params_

# X, y = example_5_dgp(200)
# best_params = get_mlp_hyperparams(X, y)
# print("Best hyperparameters:", best_params)

In [ ]:
# def example_5_f_factory(X, y):
#     # train a neural network regressor
#     mlp = MLPRegressor(
#         hidden_layer_sizes=(80,),
#         activation='relu',
#         solver='adam',
#         alpha=0.00001,
#         learning_rate='constant',
#         max_iter=1000,
#         random_state=42
#     )
#     mlp.fit(X, y)
#     return lambda X: mlp.predict(X)

# n = 200
# bins = bin_selection(n)

# vim_mains, vim_connected, vim_quantile = replicate_ale_vim_training(example_5_dgp, example_5_f_factory, 2, n=n, bins=bins, replications=100, categorical=False)

## Example 6

In [ ]:
def example_6_dgp(n):
    # generate copula of uniform random variables
    # with specific correlation structure
    R = np.array([
        [1.0, 0.0, 0.2, 0.0],
        [0.0, 1.0, 0.9, 0.0],
        [0.2, 0.9, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ])

    L = np.linalg.cholesky(R)
    Z = np.random.normal(size=(n, 4)) @ L.T
    X = stats.norm.cdf(Z)
    y = 4 * X[:, 0] + 3.87 * X[:, 1] + 2.97 * np.exp(-5 + 10 * X[:, 2]) / (1 + np.exp(-5 + 10 * X[:, 2])) + 13.86 * X[:, 0] * X[:, 1] + np.random.normal(0, 0.5, X.shape[0])
    return X, y

def example_6_f_factory_mlp(X, y):
    pass

def example_6_f_factory_gradient_boosted_tree(X, y):
    pass

def example_6_f_factory_random_forest(X, y):
    pass

n = 10000
dgp = lambda n: example_6_dgp(n)
bins = bin_selection(n)

vim_mains, vim_connected, vim_quantile = replicate_ale_vims(dgp, example_6_f_factory_mlp, nvars=4, n=n, bins=bins, replications=1, categorical=False)

In [ ]:
vim_mains, vim_connected, vim_quantile = replicate_ale_vims(dgp, example_6_f_factory_mlp, nvars=4, n=n, bins=bins, replications=1, categorical=False)

In [ ]:
vim_mains, vim_connected, vim_quantile = replicate_ale_vims(dgp, example_6_f_factory_mlp, nvars=4, n=n, bins=bins, replications=1, categorical=False)

## Example 7

In [ ]:
# clean data
df = pd.read_csv("data/uci_bikesharing.csv")
df = df.dropna()
df["quarter"] = 1 + 4 * df["yr"] + df["mnth"] // 4
# rename mnth to month, hr to hour
df = df.rename(columns={"mnth": "month", "hr": "hour", "weathersit": "weather_situation"})
# keep only relevant columns
X = df[["quarter", "month", "hour", "holiday", "weekday", "workingday", "weather_situation", "atemp", "hum", "windspeed"]]
y = df["cnt"]

In [ ]:
# correlation matrix of covariates
corr = X.corr()
plt.figure(figsize=(10, 8))
plt.matshow(corr, cmap='coolwarm', fignum=1)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.title('Correlation Matrix of Covariates', pad=20)
plt.show()

In [ ]:
def train_regressor(X_train, y_train):
    # train neural network with 1 hidden layer (35 nodes), relu activation, and 0.05 L2 regularization
    # NOTE: 35 nodes and 0.05 alpha were chosen based on previous cross-validation results in Apley and Zhu
    # perform log-transform on target variable to handle skewness
    model = TransformedTargetRegressor(
        regressor=MLPRegressor(hidden_layer_sizes=(35,), activation='relu', alpha=0.05, max_iter=1000, random_state=42),
        func=np.log,
        inverse_func=np.exp
    )
    
    model.fit(X_train, y_train)
    return model

In [ ]:
def summarize_model_performance(model, X_test, y_test):
    # evaluate model performance
    y_pred = model.predict(X_test)
    y_pred = np.exp(y_pred)  # convert back to original scale
    y_true = np.exp(y_test)  # convert back to original scale
    mse = mean_squared_error(y_true, y_pred)
    print("Mean Squared Error:", mse)
    print("R^2:", model.score(X_test, y_test))

In [ ]:
est = make_pipeline(
    TransformedTargetRegressor(
        regressor=MLPRegressor(hidden_layer_sizes=(5,), activation='relu', alpha=0.05, max_iter=1000, random_state=42),
        func=np.log,
        inverse_func=np.exp
    )
)
# TODO: find out why R^2 is so low (0.80, expect 0.93)

# cv = KFold(n_splits=5, shuffle=True, random_state=42)
# scores = cross_val_score(est, X, y, cv=cv, scoring='r2')
# print("Cross-validated R^2 scores:", scores)



In [ ]:
covariate_names = X.columns.tolist()
X_list = X[:1600].to_numpy() # make round number for binning
y_list = y[:1600].to_numpy()
n = X_list.shape[0]
K = bin_selection(n)
est.fit(X_list, y_list)
# function to plug-in representing model
f = lambda x: est.predict(x)

In [ ]:
covariate_names

In [ ]:
categorical_features = ["quarter", "month", "hour", "holiday", "weekday", "workingday", "weather_situation"]

In [ ]:
for i, predictor in enumerate(covariate_names):
    print(f"Feature: {predictor}")
    if predictor in categorical_features:
        continue
    categorical = predictor in categorical_features
    main = ale_main_vim(f, X_list, i + 1, K, categorical=categorical)
    connected = ale_connected_total(f, X_list, i + 1, K, categorical=categorical)
    quantile = ale_quantile_total(f, X_list, i + 1, K, categorical=categorical)
    print(f"Main: {main}, Connected Total: {connected}, Quantile Total: {quantile}\n")